## 08b_lenormand_graph_export_levels
  Exports the observed commuting network at every level of notebook 09 as graphml.
  Input: data/interim/09_lenormand_multilevel/. Output: data/processed/08b_lenormand_graph_export_levels/.


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# --- find the repo root (the folder containing data/interim) and work from there ---
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "interim").is_dir())
os.chdir(ROOT)

# ----------------------------- CONFIG -----------------------------
IN_DIR = Path("data/interim/09_lenormand_multilevel")
OUT_DIR = Path("data/processed/08b_lenormand_graph_export_levels")
LEVELS = [("CZ", "kraj"), ("PL", "nuts3"), ("CZ", "okres"), ("PL", "powiat"),
          ("CZ", "orp"), ("PL", "gmina"), ("CZ", "obec")]      # small to large
# ------------------------------------------------------------------
OUT_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for country, level in LEVELS:
    units = pd.read_csv(IN_DIR / f"{country}_{level}_units.csv")
    od = np.load(IN_DIR / f"{country}_{level}_od_observed.npy", mmap_mode="r")   # read only, not loaded
    rows.append({"country": country, "level": level, "n_nodes": len(units),
                 "n_edges": int(np.count_nonzero(od)), "total_flow": float(od.sum()),
                 "possible_edges": len(units) * (len(units) - 1)})
sizes = pd.DataFrame(rows)
sizes["edge_density"] = (sizes["n_edges"] / sizes["possible_edges"]).round(3)
sizes.to_csv(OUT_DIR / "levels_graph_size.csv", index=False)
print(sizes.to_string(index=False))
print("saved:", OUT_DIR / "levels_graph_size.csv")

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx

# --- find the repo root (the folder containing data/interim) and work from there ---
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "interim").is_dir())
os.chdir(ROOT)

# ----------------------------- CONFIG -----------------------------
IN_DIR = Path("data/interim/09_lenormand_multilevel")
OUT_DIR = Path("data/processed/08b_lenormand_graph_export_levels")
LEVELS = [("CZ", "kraj"), ("PL", "nuts3"), ("CZ", "okres"), ("PL", "powiat"),
          ("CZ", "orp"), ("PL", "gmina"), ("CZ", "obec")]      # small to large
OVERWRITE = True                                              # rewrite existing files (adds the names)
NODE_COLS = {"population": "population", "area_km2": "area_km2", "x_m": "x_m", "y_m": "y_m"}
SUMMARY_FILE = OUT_DIR / "graph_files.csv"
# name sources: GISCO NUTS file (as in notebook 08) and the LAU input tables of notebook 09
GISCO_NUTS = Path("data/raw/nuts3/NUTS_RG_03M_2024_4326.geojson")
GISCO_COL_ID, GISCO_COL_NAME = "NUTS_ID", "NUTS_NAME"
CZ_OBEC_INPUTS = Path("data/interim/cz_obec_inputs.parquet")
PL_GMINA_INPUTS = Path("data/interim/pl_gmina_inputs.parquet")
PL_CODE_LEN = 6                                               # gmina code length (leading zeros)
# ------------------------------------------------------------------
OUT_DIR.mkdir(parents=True, exist_ok=True)
graph_path = lambda country, level: OUT_DIR / f"{country}_{level}_observed.graphml"

# --- name tables per (country, level): unit_id -> name; levels without a source get no names ---
with open(GISCO_NUTS, encoding="utf-8") as f:
    props = pd.DataFrame([ft["properties"] for ft in json.load(f)["features"]])
nuts_names = dict(zip(props[GISCO_COL_ID].astype(str), props[GISCO_COL_NAME]))
cz_obec = pd.read_parquet(CZ_OBEC_INPUTS, columns=["LAU_ID", "LAU_NAME"])
pl_gmina = pd.read_parquet(PL_GMINA_INPUTS, columns=["p6", "LAU_NAME"])
NAME_SOURCES = {
    ("CZ", "kraj"): nuts_names,
    ("PL", "nuts3"): nuts_names,
    ("CZ", "obec"): dict(zip(cz_obec["LAU_ID"].astype(str), cz_obec["LAU_NAME"])),
    ("PL", "gmina"): dict(zip(pl_gmina["p6"].astype(str).str.zfill(PL_CODE_LEN), pl_gmina["LAU_NAME"])),
}

rows = []
for country, level in LEVELS:
    f = graph_path(country, level)
    units = pd.read_csv(IN_DIR / f"{country}_{level}_units.csv", dtype={"unit_id": str})
    ids = units["unit_id"].to_numpy()
    names = [NAME_SOURCES.get((country, level), {}).get(i, "") for i in ids]
    n_no_name = sum(1 for n in names if not n)
    if f.exists() and not OVERWRITE:
        print("Exists, skipped:", f)
    else:
        od = np.load(IN_DIR / f"{country}_{level}_od_observed.npy")
        src, dst = np.nonzero(od)                      # observed links, diagonal is zero
        G = nx.DiGraph(country=country, level=level, variant="observed")
        G.add_nodes_from((ids[i], {"name": str(names[i]),
                                   **{a: float(units[c].iloc[i]) for a, c in NODE_COLS.items()}})
                         for i in range(len(ids)))
        G.add_weighted_edges_from(zip(ids[src], ids[dst], od[src, dst].astype(float)))
        nx.write_graphml(G, f)
        print(f"Saved: {f} | nodes: {G.number_of_nodes()} | edges: {G.number_of_edges()}"
              f" | nodes without name: {n_no_name}")
        del od, src, dst, G
    rows.append({"country": country, "level": level, "file": str(f),
                 "size_mb": round(f.stat().st_size / 1e6, 2), "n_nodes": len(ids),
                 "n_nodes_without_name": n_no_name})

summary = pd.DataFrame(rows).merge(pd.read_csv(OUT_DIR / "levels_graph_size.csv")
                                   [["country", "level", "n_edges"]], on=["country", "level"])
summary.to_csv(SUMMARY_FILE, index=False)
print(summary[["country", "level", "n_nodes", "n_edges", "size_mb", "n_nodes_without_name"]]
      .to_string(index=False))
print("saved:", SUMMARY_FILE)